# 06 · Private public-corpus RAG evaluation
Exact quote support is a proxy, not factual truth, relevance or completeness.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print('Repository:', ROOT.name)

In [ ]:
from portshield_ai.rag import load_corpus, answer, groundedness
docs = load_corpus('data/public/corpus.json')
cases = [('public data opening plan', False), ('model serving quantization memory', False), ('secret classified facility topology', True)]
results=[]
for question, expected_abstain in cases:
    result=answer(question,docs,['analyst'])
    results.append({'question':question,'expected_abstain':expected_abstain,'abstained':result['abstained'],'quote_support_proxy':groundedness(result,docs)})
results

In [ ]:
assert answer('public data opening plan',docs,[])['abstained']
Path('results/runs').mkdir(parents=True,exist_ok=True)
Path('results/runs/rag-extractive.json').write_text(json.dumps(results,indent=2))

In [ ]:
if os.getenv('PORTSHIELD_RUN_SERVING') == '1':
    from portshield_ai.inference import VLLMClient
    with_client=VLLMClient()
    try:
        display(answer('public data opening plan',docs,['analyst'],with_client))
    finally:
        with_client.close()
else:
    print('Generative quote selection not run.')

Ask independent reviewers to rate evidence relevance, currentness and sufficiency. Expand holdout with no-evidence questions and poisoned/injected documents. Include negative cases in denominator; never treat a null support score as perfect groundedness.